# Vision 실습 문제 (Practice)

각 셀에서 `## Write Answer` 부분을 채워보세요.

## 출제 범위
- **데이터 구성 & Library 활용**: torchvision transforms로 학습/평가 데이터 파이프라인 구성
- **평가 Metric**: Top-1 Accuracy, Confusion Matrix 계산 방법
- **모델 원리(ResNet/ViT/DETR/U-Net)**: 주요 레이어 구성, 동작 원리, Skip/Attention/Pooling 등
- **학습 원리**: Optimizer, 학습 루프 3단계(zero_grad/backward/step)

## 문제 1. CIFAR-10 학습용 Transform 구성

[문제]
Q 1) CIFAR-10 학습을 위한 데이터 전처리 및 증강 파이프라인(train_tfms)을 구성합니다.

[요구사항]
- torchvision.transforms.Compose를 활용하여 변환 파이프라인 구성
- 32x32 크기(패딩 4) 무작위 자르기(RandomCrop) 적용
- 50% 확률의 무작위 수평 반전(RandomHorizontalFlip) 적용
- PIL 이미지를 파이토치 텐서로 변환(ToTensor)
- 주어진 CIFAR10_MEAN, CIFAR10_STD 통계치를 이용한 텐서 정규화(Normalize) 적용

In [ ]:
# [코드]
import torchvision.transforms as T

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)

train_tfms = T.Compose([
    T.### 작성 필요 ###(32, padding=4),
    T.### 작성 필요 ###(),
    T.### 작성 필요 ###(),
    T.### 작성 필요 ###(CIFAR10_MEAN, CIFAR10_STD),
])

## 문제 2. Top-1 Accuracy 계산

[문제]
Q 2) 로짓(logits)과 타깃(targets)으로부터 배치 Top-1 정확도를 계산하는 accuracy_top1 함수를 작성합니다.

[요구사항]
- 로짓 텐서의 클래스 축 기준 최댓값을 갖는 인덱스를 예측 클래스로 도출
- 정답 레이블과 일치하는 비율을 계산하여 파이썬 float 스칼라 값으로 반환

In [ ]:
# [코드]
def accuracy_top1(logits, targets):
    preds = ### 작성 필요 ###
    return (preds == targets).float().mean().item()

## 문제 3. Confusion Matrix 집계

[문제]
Q 3) 모델의 예측 클래스와 실제 정답 레이블로부터 Confusion Matrix(cm)를 집계합니다.

[요구사항]
- Confusion Matrix는 행(row)이 정답 레이블, 열(column)이 예측 클래스인 C x C 형태의 카운트 행렬
- 배치 내 각 샘플의 (정답, 예측) 쌍 위치에 카운트 1씩 누적

In [ ]:
# [코드]
import torch

def confusion_and_perclass(preds, labels, num_classes):
    cm = torch.zeros((num_classes, num_classes), dtype=torch.int64)
    for t, p in zip(labels, preds):
        cm[### 작성 필요 ###, ### 작성 필요 ###] += 1
    per_class_acc = cm.diagonal().float() / torch.clamp(cm.sum(dim=1).float(), min=1.0)
    return cm, per_class_acc

## 문제 4. 학습 루프의 핵심 3단계

[문제]
Q 4) PyTorch 표준 학습 루프 내에서 가중치를 갱신하는 3단계 핵심 코드를 순서대로 완성합니다.

[요구사항]
- 1단계: 이전 배치의 기울기(gradient) 버퍼를 초기화
- 2단계: 손실 값에 대한 역전파(backpropagation)를 수행하여 기울기 계산
- 3단계: 옵티마이저를 통해 모델의 파라미터를 1단계 갱신

In [ ]:
# [코드]
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        ### 작성 필요 ###
        logits = model(images)
        loss = criterion(logits, labels)
        ### 작성 필요 ###
        ### 작성 필요 ###

## 문제 5. SGD + Momentum Optimizer

[문제]
Q 5) 모멘텀(Momentum)과 가중치 감쇠(Weight Decay)가 적용된 SGD 옵티마이저를 초기화합니다.

[요구사항]
- torch.optim.SGD를 사용하고 모델 파라미터 전달
- 학습률 lr=0.1, 모멘텀 계수 momentum=0.9, 가중치 감쇠 weight_decay=5e-4 설정

In [ ]:
# [코드]
import torch
import torch.nn as nn

criterion = nn.CrossEntropyLoss()
optimizer = ### 작성 필요 ###

## 문제 6. ViT PreNorm (LayerNorm)

[문제]
Q 6) Transformer의 Pre-LayerNorm 래퍼 모듈(PreNorm)을 구현합니다.

[요구사항]
- nn.LayerNorm을 지정된 차원(dim)으로 초기화
- forward(x, **kwargs)에서 입력 x에 대해 LayerNorm을 먼저 적용한 후 서브모듈(fn)에 전달

In [ ]:
# [코드]
import torch.nn as nn

class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = ### 작성 필요 ###
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)

## 문제 7. ViT FeedForward (Linear → GELU → Linear)

[문제]
Q 7) Vision Transformer의 FeedForward (MLP) 블록을 구성합니다.

[요구사항]
- 차원 확장 선형층(dim -> hidden_dim), GELU 비선형 활성화 함수, 드롭아웃 레이어, 차원 복원 선형층(hidden_dim -> dim), 드롭아웃 레이어 순으로 구성
- nn.Sequential을 사용하여 일련의 레이어를 연결

In [ ]:
# [코드]
import torch.nn as nn

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.):
        super().__init__()
        self.net = nn.Sequential(
            nn.### 작성 필요 ###,
            nn.### 작성 필요 ###,
            nn.Dropout(dropout),
            nn.### 작성 필요 ###,
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

## 문제 8. Multi-Head Attention inner_dim 계산

[문제]
Q 8) Multi-Head Attention 초기화 시 각 헤드의 차원(dim_head)과 총 헤드 수(heads)로부터 내부 프로젝션 차원(inner_dim)을 계산합니다.

[요구사항]
- inner_dim은 각 헤드의 차원(dim_head)과 헤드 개수(heads)의 곱으로 산출

In [ ]:
# [코드]
import torch.nn as nn

class AttentionInit(nn.Module):
    def __init__(self, dim, heads=4, dim_head=64):
        super().__init__()
        inner_dim = ### 작성 필요 ###
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)
        self.to_out = nn.Linear(inner_dim, dim)

## 문제 9. Scaled Dot-Product Attention (einsum)

[문제]
Q 9) einsum 표기법을 활용하여 Scaled Dot-Product Attention의 유사도 및 어텐션 가중치를 계산합니다.

[요구사항]
- Q와 K의 내적을 einsum으로 계산하고 스케일 팩터(self.scale)를 적용하여 dots 산출
- 마지막 축(dim=-1)에 softmax를 취하여 attention 가중치 확률 분포 생성

In [ ]:
# [코드]
import torch
from torch import nn, einsum
from einops import rearrange

class Attention(nn.Module):
    def __init__(self, dim, heads=4, dim_head=64):
        super().__init__()
        inner_dim = dim_head * heads
        self.heads = heads
        self.scale = dim_head ** -0.5
        self.attend = nn.Softmax(dim=-1)
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)
        self.to_out = nn.Linear(inner_dim, dim)

    def forward(self, x):
        b, n, _, h = *x.shape, self.heads
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=h), qkv)

        dots = ### 작성 필요 ###
        attn = ### 작성 필요 ###

        out = einsum('b h i j, b h j d -> b h i d', attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out)

## 문제 10. Transformer Block 구성

[문제]
Q 10) Multi-Head Attention과 FeedForward를 결합한 Transformer Block의 레이어 리스트를 구성합니다.

[요구사항]
- 각 트랜스포머 레이어는 PreNorm으로 감싼 Attention 블록과 PreNorm으로 감싼 FeedForward 블록으로 구성
- ※ 사전 정의 컴포넌트: PreNorm(dim, fn), Attention(dim, heads, dim_head, dropout), FeedForward(dim, mlp_dim, dropout) 가 제공됩니다.

In [ ]:
# [코드]
import torch.nn as nn

class Transformer(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout=0.):
        super().__init__()
        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                ### 작성 필요 ###,
                ### 작성 필요 ###,
            ]))

    def forward(self, x):
        for attn, ff in self.layers:
            x = attn(x) + x
            x = ff(x) + x
        return x

## 문제 11. ViT 패치 임베딩 (einops Rearrange 패턴)

[문제]
Q 11) einops Rearrange를 사용하여 (B, C, H, W) 이미지를 평탄화된 패치 시퀀스 (B, num_patches, patch_dim)로 재배열하는 레이어를 작성합니다.

[요구사항]
- 패치 높이(p1)와 패치 너비(p2)를 기준으로 공간 차원을 분할하고 채널과 결합하여 1D 토큰 차원으로 평탄화

In [ ]:
# [코드]
from einops.layers.torch import Rearrange
import torch.nn as nn

patch_height = 4
patch_width = 4
patch_dim = 3 * patch_height * patch_width
dim = 64

to_patch_embedding = nn.Sequential(
    Rearrange(### 작성 필요 ###, p1=patch_height, p2=patch_width),
    nn.Linear(patch_dim, dim),
)

## 문제 12. ViT CLS 토큰 결합 & 위치 임베딩

[문제]
Q 12) 패치 임베딩 시퀀스 앞에 학습 가능한 [CLS] 토큰을 결합하고 위치 임베딩(pos_embedding)을 더합니다.

[요구사항]
- 배치 크기(b)만큼 복제된 [CLS] 토큰 텐서를 패치 텐서의 시퀀스 축(dim=1) 맨 앞에 결합
- 결합된 전체 시퀀스 길이에 맞추어 위치 임베딩 텐서를 슬라이싱한 후 덧셈 연산 수행

In [ ]:
# [코드]
import torch
from torch import nn
from einops import repeat

class ViTForward(nn.Module):
    def __init__(self, num_patches, dim):
        super().__init__()
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, dim))

    def forward(self, x):
        b, n, _ = x.shape
        cls_tokens = repeat(self.cls_token, ### 작성 필요 ###, b=b)
        x = torch.cat(### 작성 필요 ###)
        x = x + self.pos_embedding[:, :(### 작성 필요 ###)]
        return x

## 문제 13. ViT 분류 표현 추출: CLS 토큰 vs mean pooling

[문제]
Q 13) ViT 인코더 출력에서 pool 모드 설정('cls' 또는 'mean')에 따라 분류 헤드로 전달할 최종 표현 벡터를 추출합니다.

[요구사항]
- pool == 'cls'인 경우 시퀀스의 맨 첫 번째 [CLS] 토큰 벡터 추출
- 그 외('mean')인 경우 시퀀스 축(dim=1) 전체에 대한 평균 벡터 산출

In [ ]:
# [코드]
def select_feature(x, pool):
    x = ### 작성 필요 ###
    return x

## 문제 14. DETR 입력 전처리 파이프라인

[문제]
Q 14) 사전학습 DETR 모델 입력을 위해 ResNet 표준 전처리 파이프라인(Resize, ToTensor, Normalize)을 구성합니다.

[요구사항]
- 짧은 변 기준 800픽셀 리사이즈(T.Resize)
- 텐서 변환(T.ToTensor) 및 ImageNet 표준 평균/표준편차 기반 정규화(T.Normalize) 구성

In [ ]:
# [코드]
import torchvision.transforms as T

transform = T.Compose([
    T.### 작성 필요 ###(800),
    T.### 작성 필요 ###(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

## 문제 15. 박스 좌표 변환 (cx,cy,w,h) → (xmin,ymin,xmax,ymax)

[문제]
Q 15) 바운딩 박스 중심 좌표 표현 (cx, cy, w, h)을 모서리 좌표 표현 (xmin, ymin, xmax, ymax)으로 변환하는 함수를 작성합니다.

[요구사항]
- 입력 텐서의 중심(cx, cy)과 폭/높이(w, h)를 언패킹
- xmin = cx - 0.5*w, ymin = cy - 0.5*h, xmax = cx + 0.5*w, ymax = cy + 0.5*h 계산 후 마지막 차원 기준으로 결합

In [ ]:
# [코드]
import torch

def box_cxcywh_to_xyxy(x):
    x_c, y_c, w, h = x.unbind(1)
    b = [(x_c - 0.5 * w), (y_c - 0.5 * h),
         (x_c + 0.5 * w), (y_c + 0.5 * h)]
    return ### 작성 필요 ###

## 문제 16. 정규화 박스를 픽셀 좌표로 복원

[문제]
Q 16) [0, 1] 범위로 정규화된 바운딩 박스 좌표를 원본 이미지 픽셀 크기(w, h)에 맞게 역스케일링합니다.

[요구사항]
- 각 좌표 (xmin, ymin, xmax, ymax)에 곱해질 [w, h, w, h] 크기의 배율 텐서 구성 및 곱셈 수행
- ※ 사전 정의 함수: box_cxcywh_to_xyxy(x) 가 제공됩니다.

In [ ]:
# [코드]
import torch

def rescale_bboxes(out_bbox, size):
    img_w, img_h = size
    b = box_cxcywh_to_xyxy(out_bbox)
    b = b * torch.tensor(### 작성 필요 ###, dtype=torch.float32)
    return b

## 문제 17. DETR 예측 신뢰도 필터링

[문제]
Q 17) DETR 출력 로짓으로부터 no-object 클래스를 제외한 최대 예측 확률을 구하고 신뢰도 임계값(0.7)으로 필터링 마스크를 생성합니다.

[요구사항]
- 마지막 차원에 소프트맥스를 취한 뒤 no-object(마지막 클래스)를 제외한 영역에서 최댓값 및 해당 인덱스 추출
- 최대 예측 확률이 0.7 이상인 인덱스 위치를 나타내는 불리언 마스크(keep) 생성

In [ ]:
# [코드]
probas = outputs['pred_logits'].softmax(### 작성 필요 ###)[0, :, :### 작성 필요 ###]
keep = ### 작성 필요 ###

## 문제 18. DETR keep 마스크로 박스 선택

[문제]
Q 18) 신뢰도 필터링 마스크(keep)를 적용하여 기준을 통과한 유효 바운딩 박스들만 선택하고 원본 이미지 크기로 복원합니다.

[요구사항]
- outputs['pred_boxes']의 배치 0번에서 keep 마스크에 해당하는 박스들만 슬라이싱
- ※ 사전 정의 함수: rescale_bboxes(out_bbox, size) 가 제공됩니다.

In [ ]:
# [코드]
bboxes_scaled = rescale_bboxes(outputs['pred_boxes'][0, ### 작성 필요 ###], im.size)

## 문제 19. U-Net DoubleConv 블록

[문제]
Q 19) U-Net의 기본 구성 단위인 DoubleConv 블록(3x3 Conv -> BatchNorm -> ReLU 2회 반복)을 완성합니다.

[요구사항]
- 첫 번째 Conv는 in_channels -> out_channels, 두 번째 Conv는 out_channels -> out_channels 적용
- 공간 크기 보존을 위해 kernel_size=3, padding=1 사용

In [ ]:
# [코드]
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if mid_channels is None:
            mid_channels = out_channels
        self.net = nn.Sequential(
            ### 작성 필요 ###,
            ### 작성 필요 ###,
            ### 작성 필요 ###,
            ### 작성 필요 ###,
        )

    def forward(self, x):
        return self.net(x)

## 문제 20. U-Net Down 블록 (MaxPool)

[문제]
Q 20) U-Net의 다운샘플링 블록(Down)에서 피처맵 크기를 절반으로 축소하는 풀링 레이어를 정의합니다.

[요구사항]
- 커널 크기 2, 스트라이드 2의 nn.MaxPool2d 레이어 사용
- ※ 사전 정의 컴포넌트: DoubleConv(in_channels, out_channels) 가 제공됩니다.

In [ ]:
# [코드]
import torch.nn as nn

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.pool = ### 작성 필요 ###
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x):
        x = self.pool(x)
        x = self.conv(x)
        return x

## 문제 21. U-Net Up 블록 (Upsample / ConvTranspose2d)

[문제]
Q 21) U-Net의 업샘플링 블록(Up)에서 bilinear 모드에 따라 Upsample 또는 ConvTranspose2d 레이어를 생성합니다.

[요구사항]
- bilinear=True인 경우 nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True) 사용
- bilinear=False인 경우 nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2) 사용
- ※ 사전 정의 컴포넌트: DoubleConv(in_channels, out_channels) 가 제공됩니다.

In [ ]:
# [코드]
import torch.nn as nn

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=False):
        super().__init__()
        if bilinear:
            self.up = ### 작성 필요 ###
            self.conv = DoubleConv(in_channels, out_channels, mid_channels=in_channels // 2)
        else:
            self.up = ### 작성 필요 ###
            self.conv = DoubleConv(in_channels, out_channels)

## 문제 22. U-Net Skip Connection (채널 방향 concat)

[문제]
Q 22) U-Net의 Up 블록 forward에서 스킵 연결(Skip Connection) 텐서(x2)와 현재 피처맵(x1)을 채널 축 기준으로 결합합니다.

[요구사항]
- 두 텐서를 채널 축(dim=1) 기준으로 torch.cat 결합하여 공간적 세부 정보 복원
- ※ 사전 정의 컴포넌트: DoubleConv(in_channels, out_channels) 가 제공됩니다.

In [ ]:
# [코드]
import torch
import torch.nn as nn
import torch.nn.functional as F

class UpForward(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diff_y = x2.size(2) - x1.size(2)
        diff_x = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [diff_x // 2, diff_x - diff_x // 2,
                        diff_y // 2, diff_y - diff_y // 2])
        x = ### 작성 필요 ###
        return self.conv(x)

## 문제 23. U-Net 전체 구조 (입력 stem & 디코더 Skip Connection 호출)

[문제]
Q 23) U-Net 전체 네트워크에서 입력 stem 합성곱 및 디코더의 스킵 연결 호출부를 완성합니다.

[요구사항]
- 입력 이미지 채널(n_channels)을 64 채널로 변환하는 DoubleConv 초기 stem 정의
- 각 Up 레이어 호출 시 이전 디코더 출력과 인코더의 대칭되는 스킵 연결 피처맵(x4, x3, x2, x1)을 순서대로 전달
- ※ 사전 정의 컴포넌트: DoubleConv(in_c, out_c), Down(in_c, out_c), Up(in_c, out_c, bilinear) 가 제공됩니다.

In [ ]:
# [코드]
import torch.nn as nn

class UNet(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=False):
        super().__init__()
        self.inc = DoubleConv(### 작성 필요 ###, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(### 작성 필요 ###, ### 작성 필요 ###)
        x = self.up2(### 작성 필요 ###, ### 작성 필요 ###)
        x = self.up3(### 작성 필요 ###, ### 작성 필요 ###)
        x = self.up4(### 작성 필요 ###, ### 작성 필요 ###)
        return self.outc(x)

## 문제 24. Object Detection 평가 지표: IoU (Intersection over Union) 계산

[문제]
Q 24) 두 바운딩 박스 [x1, y1, x2, y2]의 교집합 넓이와 합집합 넓이를 구해 IoU(Intersection over Union)를 계산합니다.

[요구사항]
- 교집합 좌상단 좌표는 두 박스의 최댓값(max), 우하단 좌표는 두 박스의 최솟값(min)으로 계산
- 음수 폭/높이를 0으로 클램핑(max(0.0, ...))하여 유효한 교집합 넓이(inter_area) 산출
- 두 박스의 넓이 합에서 inter_area를 뺀 합집합 넓이로 나누어 IoU 반환

In [ ]:
# [코드]
def compute_iou(box1, box2):
    # box1, box2: [x1, y1, x2, y2]
    inter_x1 = ### 작성 필요 ###
    inter_y1 = ### 작성 필요 ###
    inter_x2 = ### 작성 필요 ###
    inter_y2 = ### 작성 필요 ###

    inter_area = ### 작성 필요 ###
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = ### 작성 필요 ###
    return inter_area / (union_area + 1e-6)